In [ ]:
from __future__ import annotations
from collections.abc import Sequence
from functools import total_ordering
from itertools import permutations
from math import cos, pi, sin, floor
from random import sample, randint, shuffle

import matplotlib.pyplot as plt
from icecream import ic

ic.configureOutput(outputFunction=print)

NUM_CHILDREN = 10
NUM_CHARACTERISTICS = 20

In [ ]:
@total_ordering
class Child:
    _name: str
    _characteristics: tuple[bool, ...]

    def __init__(self, name: str, characteristics: Sequence[bool]):
        self._name = name
        self._characteristics = tuple(characteristics)

    @property
    def name(self) -> str:
        return self._name

    def difference(self, other: Child) -> int:
        return sum(s != o for s, o in zip(self._characteristics, other._characteristics))

    def __repr__(self) -> str:
        return f"{self._name}:{''.join('+' if c else '-' for c in self._characteristics)}"

    def __eq__(self, other):
        return id(self) == id(other)

    def __lt__(self, other):
        return id(self) < id(other)


In [ ]:
CHILDREN = list()

for c in range(NUM_CHILDREN):
    name = f"Child{c}"
    CHILDREN.append(
        Child(
            name,
            sample(
                [True, False], k=NUM_CHARACTERISTICS, counts=[NUM_CHARACTERISTICS, NUM_CHARACTERISTICS]
            ),
        )
    )

ic(CHILDREN)

In [ ]:
def edges(solution: Sequence[Child]) -> list[tuple[Child, Child]]:
    if len(solution) != NUM_CHILDREN:
        circle = [CHILDREN[0]] + list(solution)
    else:
        circle = list(solution)
    return list(zip(circle, circle[1:] + circle[:1]))


def fitness(solution: Sequence[Child]) -> int:
    return sum(a.difference(b) for a, b in edges(solution))


def normalize(solution: Sequence[Child]) -> list[Child]:
    solution = list(solution)
    if solution[0] > solution[-1]:
        solution.reverse()
    return solution


def tweak(solution: Sequence[Child]) -> list[Child]:
    i1, i2 = sample(range(len(solution)), k=2)
    new_solution = list(solution)
    new_solution[i1] = solution[i2]
    new_solution[i2] = solution[i1]
    return normalize(new_solution)

# Inver Over Crossover
def crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> Sequence[Child]:
    p1_gene_idx = randint(0, NUM_CHILDREN - 1)
    p1_gene = parent1[p1_gene_idx]
    p2_gene_idx = (parent2.index(p1_gene) + 1) % NUM_CHILDREN
    p2_gene = parent2[p2_gene_idx]

    idx = parent1.index(p2_gene)

    start_idx = min(p1_gene_idx, idx)
    end_idx = max(p1_gene_idx, idx)

    new_sol = parent1.copy()

    first_part = new_sol[:start_idx + 1]
    to_rev = new_sol[start_idx + 1 : end_idx + 1]
    to_rev.reverse()

    if end_idx + 1 >= NUM_CHILDREN:
        last_part = []
    else:
        last_part = new_sol[end_idx + 1:]

    new_sol = first_part +  to_rev + last_part
    new_sol = normalize(new_sol)

    return new_sol


def exhaustive_search() -> tuple[Child, ...]:
    return max(permutations(CHILDREN[1:]), key=fitness)


def hill_climbing(steps: int = 1000) -> tuple[list[Child], list[int]]:
    solution = normalize(sample(CHILDREN[1:], k=NUM_CHILDREN - 1))
    best = fitness(solution)
    history = [best]
    for _ in range(steps):
        candidate = tweak(solution)
        candidate_fitness = fitness(candidate)
        if candidate_fitness >= best:
            solution, best = candidate, candidate_fitness
        history.append(best)
    return solution, history

In [ ]:
MU = 20
LBDA = 3 * MU
GENERATION = 0

POPULATION = []

# Seeding initial population
for _ in range(MU):
    POPULATION.append(normalize(sample(CHILDREN, k=NUM_CHILDREN)))

In [ ]:
# Parent Tournament Selection
TAU = 4

def tournament_selection(population: list[list[Child]], tournament_size: int, parents_amt: int) -> list[list[Child]]:
    global TAU
    parents = []
    for _ in range(parents_amt):
        tournament = sample(population, k = TAU)
        parents.append(max(tournament, key = fitness))
    return parents

# Applying genetic operators: some parents mutate and some recombinate
def generate_offspring(population: list[list[Child]], fraction_to_mutate: float) -> list[list[Child]]:
    global LBDA, TAU
    offspring = []
    
    amt_to_mutate = floor(fraction_to_mutate * LBDA)
    amt_to_crossover = LBDA - amt_to_mutate

    if amt_to_crossover % 2 != 0:
        amt_to_crossover -= 1
        amt_to_mutate += 1

    parents_amt = amt_to_mutate + 2 * amt_to_crossover
    parents = tournament_selection(population, TAU, parents_amt)
    shuffle(parents)

    to_mutate = parents[:amt_to_mutate]

    for parent in to_mutate:
        offspring.append(tweak(parent))

    idx = amt_to_mutate
    while idx < parents_amt - 1:
        parent1 = parents[idx]
        idx = idx + 1
        parent2 = parents[idx]
        
        offspring.append(crossover(parent1, parent2))

    return offspring

def survivor_selection_by_fitness():
    global POPULATION, MU
    POPULATION.sort(key = fitness, reverse = True)
    POPULATION = POPULATION[:MU]

def advance_generation():
    global POPULATION, GENERATION
    offspring = generate_offspring(POPULATION, 0.1)

    POPULATION = POPULATION + offspring

    survivor_selection_by_fitness()
    GENERATION += 1

def genetic_algorithm(steps: int = 1000) -> tuple[list[Child], list[int]]:
    global POPULATION
    solution = max(POPULATION, key = fitness)
    best = fitness(solution)
    history = [best]
    for _ in range(steps):
        advance_generation()
        new_best = max(POPULATION, key = fitness)
        new_fitness = fitness(new_best)
        if new_fitness >= best:
            solution, best = new_best, new_fitness
        history.append(best)
    return solution, history

In [ ]:
def report(title: str, solution: Sequence[Child]) -> None:
    order = [a.name for a, _ in edges(solution)]
    distances = [a.difference(b) for a, b in edges(solution)]
    total = fitness(solution)
    ic(title, order, distances, total)

def plot_circle(ax, solution: Sequence[Child]) -> None:
    pairs = edges(solution)
    n = len(pairs)
    angles = [pi / 2 - 2 * pi * i / n for i in range(n)]
    xs = [cos(a) for a in angles]
    ys = [sin(a) for a in angles]
    for i, (a, b) in enumerate(pairs):
        j = (i + 1) % n
        ax.plot([xs[i], xs[j]], [ys[i], ys[j]], color="tab:blue", zorder=1)
        ax.text(
            (xs[i] + xs[j]) / 2,
            (ys[i] + ys[j]) / 2,
            a.difference(b),
            ha="center",
            va="center",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="tab:blue"),
            zorder=2,
        )
    ax.scatter(xs, ys, s=1800, c=["gold"] + ["white"] * (n - 1), edgecolors="black", zorder=3)
    for x, y, (a, _) in zip(xs, ys, pairs):
        ax.text(x, y, a.name, ha="center", va="center", fontsize=8, zorder=4)
    ax.set_title(f"Best circle: total distance {fitness(solution)}")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect("equal")
    ax.axis("off")


def plot_search(ax, histories: list[list[int]], optimum: Sequence[Child] | None) -> None:
    for history in histories:
        ax.plot(history, alpha=0.6)
    if optimum is not None:
        ax.axhline(fitness(optimum), color="red", linestyle="--", label="exhaustive optimum")
        ax.legend(loc="lower right")
    ax.set_xscale("symlog")
    ax.set_xlim(0, len(histories[0]))
    ax.set_xlabel("step (log scale)")
    ax.set_ylabel("total distance")
    ax.set_title(f"Hill climbing: {len(histories)} runs")


runs = [hill_climbing() for _ in range(10)]
histories = [history for _, history in runs]
climbed = max((solution for solution, _ in runs), key=fitness)
optimal = exhaustive_search() if NUM_CHILDREN <= 10 else None
answer = optimal if optimal is not None else climbed

if optimal is not None:
    report("Exhaustive search (guaranteed optimum)", optimal)
report("Hill climbing (best of 10 runs)", climbed)

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5))
plot_circle(left, answer)
plot_search(right, histories, optimal)
fig.tight_layout()
plt.show()

In [ ]:
def report(title: str, solution: Sequence[Child]) -> None:
    order = [a.name for a, _ in edges(solution)]
    distances = [a.difference(b) for a, b in edges(solution)]
    total = fitness(solution)
    ic(title, order, distances, total)

def plot_circle(ax, solution: Sequence[Child]) -> None:
    pairs = edges(solution)
    n = len(pairs)
    angles = [pi / 2 - 2 * pi * i / n for i in range(n)]
    xs = [cos(a) for a in angles]
    ys = [sin(a) for a in angles]
    for i, (a, b) in enumerate(pairs):
        j = (i + 1) % n
        ax.plot([xs[i], xs[j]], [ys[i], ys[j]], color="tab:blue", zorder=1)
        ax.text(
            (xs[i] + xs[j]) / 2,
            (ys[i] + ys[j]) / 2,
            a.difference(b),
            ha="center",
            va="center",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="tab:blue"),
            zorder=2,
        )
    ax.scatter(xs, ys, s=1800, c=["gold"] + ["white"] * (n - 1), edgecolors="black", zorder=3)
    for x, y, (a, _) in zip(xs, ys, pairs):
        ax.text(x, y, a.name, ha="center", va="center", fontsize=8, zorder=4)
    ax.set_title(f"Best circle: total distance {fitness(solution)}")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect("equal")
    ax.axis("off")


def plot_search(ax, histories: list[list[int]], optimum: Sequence[Child] | None) -> None:
    for history in histories:
        ax.plot(history, alpha=0.6)
    if optimum is not None:
        ax.axhline(fitness(optimum), color="red", linestyle="--", label="exhaustive optimum")
        ax.legend(loc="lower right")
    ax.set_xscale("symlog")
    ax.set_xlim(0, len(histories[0]))
    ax.set_xlabel("step (log scale)")
    ax.set_ylabel("total distance")
    ax.set_title(f"GA: {len(histories)} runs")


runs = [genetic_algorithm()]
histories = [history for _, history in runs]
climbed = max((solution for solution, _ in runs), key=fitness)
optimal = exhaustive_search() if NUM_CHILDREN <= 10 else None
answer = optimal if optimal is not None else climbed

if optimal is not None:
    report("Exhaustive search (guaranteed optimum)", optimal)
report("GA", climbed)

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5))
plot_circle(left, answer)
plot_search(right, histories, optimal)
fig.tight_layout()
plt.show()